# DiffPrep + AutoGluon

This notebook runs the pinned DiffPrep pipeline and evaluates its frozen
transformed features with AutoGluon and `IdentityFeatureGenerator`.
`split_train_val_test` is the shared outer splitter; the outer test stays
untouched until final scoring. AutoGluon is limited to 300 seconds in both
modes; smoke mode only reduces the number of datasets.


In [ ]:
# DiffPrep plus AutoGluon; H2O and TPOT are not used here.
%pip install -q "autogluon.tabular==1.5.0" "impyute>=0.0.8" "pyarrow>=15" "requests"


In [ ]:
from __future__ import annotations
import gc
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
import traceback
import warnings
from pathlib import Path

for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[variable] = "1"
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings("ignore")

RUN_MODE = "smoke"       # change to final after the one-dataset smoke run
NUM_DATASET_SHARDS = 5
DATASET_SHARD_INDEX = 0
METHOD = "diffprep_fix"
SPLIT_SEED = 42
TRAIN_SEED = 1
MAX_SAMPLES = 100_000
PARQUET_BATCH_SIZE = 4_096
AG_TIME_LIMIT = 300
AG_PRESETS = "best_quality"

KAGGLE = Path("/kaggle/working").exists()
OUTPUT_DIR = Path("/kaggle/working/diffprep_autogluon") if KAGGLE else Path("outputs/diffprep_autogluon")
TEMP_ROOT = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else OUTPUT_DIR / "temp"
REPO_DIR = TEMP_ROOT / "DiffPrep"
SOLUTION_DIR = TEMP_ROOT / "SolutionRecommendation"
CACHE_DIR = TEMP_ROOT / "openml_datagit_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
if RUN_MODE not in {"smoke", "final"} or not 0 <= DATASET_SHARD_INDEX < NUM_DATASET_SHARDS:
    raise ValueError("DATASET_SHARD_INDEX out of range")
print("AutoGluon limit:", AG_TIME_LIMIT)


In [ ]:
# Clone the exact DiffPrep fork and the repository containing the shared
# H2O evaluator utility.
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", "kaggle-experiments", "--single-branch", "https://github.com/dangvu53/DiffPrep.git", str(REPO_DIR)], check=True)
else:
    # A previous run may have left the leakage patch (or a failed patch)
    # in this disposable Kaggle checkout. Restore the exact fork before
    # applying the current patch below.
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "kaggle-experiments"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "switch", "kaggle-experiments"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/kaggle-experiments"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "clean", "-fd"], check=True)
commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if (SOLUTION_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(SOLUTION_DIR), "fetch", "origin", "feature/acorec-autodp-space"], check=True)
    subprocess.run(["git", "-C", str(SOLUTION_DIR), "switch", "feature/acorec-autodp-space"], check=True)
    subprocess.run(["git", "-C", str(SOLUTION_DIR), "pull", "--ff-only", "origin", "feature/acorec-autodp-space"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "feature/acorec-autodp-space", "--single-branch", "https://github.com/MothMalone/SolutionRecommendation.git", str(SOLUTION_DIR)], check=True)
solution_commit = subprocess.check_output(["git", "-C", str(SOLUTION_DIR), "rev-parse", "HEAD"], text=True).strip()
# Guard against a stale AutoGluon evaluator in the cloned checkout.
evaluator_path = SOLUTION_DIR / "scripts" / "autogluon_evaluator.py"
if not evaluator_path.exists():
    raise RuntimeError("Remote checkout is stale: scripts/autogluon_evaluator.py is missing. Push the AutoGluon scripts to feature/acorec-autodp-space, then restart this Kaggle session.")
evaluator_source = evaluator_path.read_text(encoding="utf-8")
if "def evaluate_autogluon_split" not in evaluator_source or "IdentityFeatureGenerator" not in evaluator_source:
    raise RuntimeError("Stale AutoGluon evaluator detected; restart the Kaggle session and rerun this cell.")
sys.path.insert(0, str(SOLUTION_DIR / "scripts"))
sys.path.insert(0, str(SOLUTION_DIR / "src"))
import importlib
importlib.invalidate_caches()
from automl_aco.data.loaders import load_gitlab_openml_dataset
from automl_aco.eval_ids import EVAL_IDS

# The upstream trainer evaluates X_test every epoch and passes it into
# pipeline initialization. Patch that behavior in this reproduction:
# DiffPrep may use train/validation only; outer test is reserved for H2O.
def patch_exact(path, old, new):
    path = Path(path)
    source = path.read_text(encoding="utf-8")
    if old in source:
        path.write_text(source.replace(old, new), encoding="utf-8")
    elif new not in source:
        raise RuntimeError(f"DiffPrep leakage patch target not found: {path}")

for pipeline_name in ("diffprep_fix_pipeline.py", "diffprep_flex_pipeline.py"):
    pipeline_path = REPO_DIR / "pipeline" / pipeline_name
    patch_exact(
        pipeline_path,
        "return df.isnull().values.sum() > 0",
        "return df is not None and df.isnull().values.sum() > 0",
    )
    patch_exact(
        pipeline_path,
        '        first_transformer.pre_cache(X_test, "test")',
        '        if X_test is not None:\n            first_transformer.pre_cache(X_test, "test")',
    )
patch_exact(
    REPO_DIR / "experiment" / "diffprep_experiment.py",
    "prep_pipeline.init_parameters(X_train, X_val, X_test)",
    "prep_pipeline.init_parameters(X_train, X_val, None)",
)
patch_exact(
    REPO_DIR / "experiment" / "diffprep_experiment.py",
    "result, best_model = diff_prep.fit(X_train, y_train, X_val, y_val, X_test, y_test)",
    "result, best_model = diff_prep.fit(X_train, y_train, X_val, y_val, None, None)",
)
patch_exact(
    REPO_DIR / "trainer" / "diffprep_trainer.py",
    "            test_loss, test_acc = self.evaluate(X_test, y_test, X_type='test', max_only=False)",
    "            if X_test is None or y_test is None:\n                test_loss, test_acc = float('nan'), float('nan')\n            else:\n                test_loss, test_acc = self.evaluate(X_test, y_test, X_type='test', max_only=False)",
)
patch_exact(
    REPO_DIR / "extract_and_save_pipeline.py",
    "prep_pipeline.init_parameters(X_train, X_val, X_test)",
    "prep_pipeline.init_parameters(X_train, X_val, None)",
)
patch_exact(
    REPO_DIR / "extract_and_save_pipeline.py",
    "'original_test_acc': result['best_test_acc'],",
    "'original_test_acc': None,",
)
print("Patched DiffPrep: no outer-test access during pipeline search")
os.chdir(REPO_DIR)
print("DiffPrep commit:", commit)
print("SolutionRecommendation commit:", solution_commit)


In [ ]:
# Exact 30-dataset ACORec test corpus supplied for the experiment.
DATASETS = [
    {"dataset_id": 1066, "name": "kc1-binary"},
    {"dataset_id": 1047, "name": "usp05"},
    {"dataset_id": 862, "name": "sleuth-ex2016"},
    {"dataset_id": 40663, "name": "calendarDOW"},
    {"dataset_id": 1054, "name": "mc2"},
    {"dataset_id": 876, "name": "fri-c1"},
    {"dataset_id": 18, "name": "mfeat-morphological"},
    {"dataset_id": 1520, "name": "robot-failures-lp5"},
    {"dataset_id": 1548, "name": "autoUniv-au4"},
    {"dataset_id": 378, "name": "ipums-la-99"},
    {"dataset_id": 1485, "name": "madelon"},
    {"dataset_id": 14, "name": "mfeat-fourier"},
    {"dataset_id": 27, "name": "colic"},
    {"dataset_id": 44956, "name": "abalone", "dataset_key": "abalone"},
    {"dataset_id": 1037, "name": "ada_prior", "dataset_key": "ada_prior"},
    {"dataset_id": 42932, "name": "avila", "dataset_key": "avila"},
    {"dataset_id": 40668, "name": "connect-4", "dataset_key": "connect-4"},
    {"dataset_id": 1471, "name": "eeg", "dataset_key": "eeg"},
    {"dataset_id": 100000, "name": "google", "dataset_key": "google", "source": "kaggle_csv"},
    {"dataset_id": 42165, "name": "house", "dataset_key": "house_prices"},
    {"dataset_id": 41001, "name": "jungle_chess", "dataset_key": "jungle_chess_2pcs_raw_endgame_complete"},
    {"dataset_id": 41671, "name": "micro", "dataset_key": "microaggregation2"},
    {"dataset_id": 1046, "name": "mozilla4", "dataset_key": "mozilla4"},
    {"dataset_id": 46597, "name": "obesity", "dataset_key": "obesity"},
    {"dataset_id": 30, "name": "page-blocks", "dataset_key": "page-blocks"},
    {"dataset_id": 802, "name": "pbcseq", "dataset_key": "pbcseq"},
    {"dataset_id": 722, "name": "pol", "dataset_key": "pol"},
    {"dataset_id": 40922, "name": "run_or_walk", "dataset_key": "Run_or_walk_information"},
    {"dataset_id": 1119, "name": "uscensus", "dataset_key": "USCensus"},
    {"dataset_id": 1497, "name": "wall-robot-nav", "dataset_key": "wall-robot-navigation"},
]

TARGET_OVERRIDES = {42932: "10", 100000: "Rating>4.2"}
IGNORE_OVERRIDES = {42932: ["train", "test"]}

positions = np.array_split(np.arange(len(DATASETS)), NUM_DATASET_SHARDS)
SHARD_DATASETS = [DATASETS[int(i)] for i in positions[DATASET_SHARD_INDEX]]
print(
    f"Shard {DATASET_SHARD_INDEX}/{NUM_DATASET_SHARDS - 1}: "
    f"{len(SHARD_DATASETS)} datasets"
)
display(pd.DataFrame(SHARD_DATASETS))


In [ ]:
def materialize_for_diffprep(spec):
    dataset_key = spec.get("dataset_key", str(spec["dataset_id"]))
    dataset_dir = REPO_DIR / "data" / dataset_key
    data_path = dataset_dir / "data.csv"
    info_path = dataset_dir / "info.json"

    # Google is synthetic (100000), so seed the canonical loader with the
    # exact frozen DiffPrep CSV when it is not attached as a Kaggle input.
    if spec.get("source") == "kaggle_csv":
        canonical_csv = CACHE_DIR / f"{int(spec['dataset_id'])}.csv"
        source_google = REPO_DIR / "data" / dataset_key / "data.csv"
        if not canonical_csv.exists() and source_google.exists():
            shutil.copyfile(source_google, canonical_csv)

    dataset = load_gitlab_openml_dataset(
        int(spec["dataset_id"]),
        cache_dir=str(CACHE_DIR),
        test_dataset_ids=[int(value) for value in EVAL_IDS],
        verbose=True,
        max_samples_if_test=MAX_SAMPLES,
    )
    if dataset is None:
        raise RuntimeError(f"Canonical loader could not load {spec['name']}")

    frame = pd.DataFrame(dataset["X"]).copy()
    frame["target"] = pd.Series(dataset["y"]).reset_index(drop=True)
    if len(frame) < 20 or frame.shape[1] < 2:
        raise ValueError(f"Insufficient usable data: {frame.shape}")

    # DiffPrep's build_data() label-encodes the target. The canonical
    # loader already emits contiguous integer labels, so this is idempotent.
    dataset_dir.mkdir(parents=True, exist_ok=True)
    frame.to_csv(data_path, index=False)
    info = {
        "label": "target",
        "dataset_id": int(spec["dataset_id"]),
        "dataset_name": spec["name"],
        "source": "canonical_acorec_loader",
        "original_rows": int(dataset.get("original_rows", len(frame))),
        "used_rows": int(len(frame)),
        "raw_features": int(frame.shape[1] - 1),
    }
    info_path.write_text(json.dumps(info, indent=2), encoding="utf-8")
    print(f"Canonical DiffPrep input {spec['name']}: {frame.shape}")
    del dataset, frame
    gc.collect()
    return dataset_key, info


In [ ]:
# The evaluator reloads and transforms the frozen pipeline after DiffPrep
# search. This cell intentionally does not fit an AutoGluon model.


In [ ]:
RESULT_PATH = OUTPUT_DIR / f"diffprep_autogluon_shard_{DATASET_SHARD_INDEX:02d}_of_{NUM_DATASET_SHARDS:02d}.csv"
RESULT_DIR = OUTPUT_DIR / "per_dataset"; RESULT_DIR.mkdir(parents=True, exist_ok=True)
rows = pd.read_csv(RESULT_PATH).to_dict("records") if RESULT_PATH.exists() else []
def upsert(row):
    key = (str(row.get("dataset_id")), str(row.get("setting")))
    rows[:] = [old for old in rows if (str(old.get("dataset_id")), str(old.get("setting"))) != key]
    rows.append(row); pd.DataFrame(rows).to_csv(RESULT_PATH, index=False)
positions = np.array_split(np.arange(len(DATASETS)), NUM_DATASET_SHARDS)
SHARD_DATASETS = [DATASETS[int(i)] for i in positions[DATASET_SHARD_INDEX]]
RUN_DATASETS = SHARD_DATASETS[:1] if RUN_MODE == "smoke" else SHARD_DATASETS
for position, spec in enumerate(RUN_DATASETS, start=1):
    key = (str(spec["dataset_id"]), "diffprep")
    if any((str(x.get("dataset_id")), x.get("setting"), x.get("status")) == (*key, "ok") for x in rows):
        print("SKIP successful:", spec["name"]); continue
    started = time.perf_counter(); output_json = RESULT_DIR / f"{int(spec['dataset_id'])}_diffprep.json"
    print(f"[{position}/{len(RUN_DATASETS)}] {spec['name']} / diffprep")
    try:
        dataset_key = spec.get("dataset_key", str(spec["dataset_id"]))
        dataset_key, _ = materialize_for_diffprep(spec)
        subprocess.run([sys.executable, "main.py", "--dataset", dataset_key, "--method", METHOD, "--model", "log", "--split_seed", str(SPLIT_SEED), "--train_seed", str(TRAIN_SEED)], cwd=REPO_DIR, check=True)
        subprocess.run([sys.executable, "extract_and_save_pipeline.py", "--dataset", dataset_key, "--method", METHOD, "--split_seed", str(SPLIT_SEED)], cwd=REPO_DIR, check=True)
        subprocess.run([sys.executable, "extract_pipeline_config.py", "--dataset", dataset_key, "--method", METHOD], cwd=REPO_DIR, check=True)
        command = [sys.executable, str(SOLUTION_DIR / "scripts/evaluate_diffprep_autogluon.py"), "--repo-dir", str(REPO_DIR), "--dataset-key", dataset_key, "--dataset-id", str(spec["dataset_id"]), "--dataset-name", spec["name"], "--method", METHOD, "--output-json", str(output_json), "--split-seed", str(SPLIT_SEED), "--train-seed", str(TRAIN_SEED), "--time-limit", str(AG_TIME_LIMIT), "--presets", AG_PRESETS]
        process = subprocess.run(command, cwd=SOLUTION_DIR, check=False)
        row = json.loads(output_json.read_text(encoding="utf-8")) if output_json.exists() else {"status": "failed", "error": f"exit code {process.returncode}"}
    except Exception as error:
        traceback.print_exc()
        row = {"status": "failed", "error_type": type(error).__name__, "error": str(error)[:4000]}
    row.update({"dataset_id": int(spec["dataset_id"]), "dataset": spec["name"], "setting": "diffprep", "notebook_wall_clock_seconds": time.perf_counter() - started})
    upsert(row); gc.collect()
display(pd.DataFrame(rows).sort_values(["dataset_id", "setting"]))
print("Saved:", RESULT_PATH)


## Outputs

The result CSV includes AutoGluon fit/prediction/total runtime and the
notebook wall clock. Successful rows must report
`diffprep_test_seen_during_search=False`.
